# Lab 03: Norms & Special Matrices

**Reference:** Goodfellow, Bengio & Courville, *Deep Learning* (2016), Chapter 2, Sections 2.5--2.6

---

**What this lab covers and why it matters:**

In machine learning, nearly every algorithm reduces to numerical operations on vectors and matrices. Two questions come up constantly:

1. **How big is this vector / matrix?** -- That is what *norms* answer (Section 2.5).
2. **Does this matrix have special structure I can exploit?** -- That is what *special matrices* address (Section 2.6).

By the end of this notebook you will be able to:
- State the three axioms that define a norm and explain why each is needed.
- Compute $L^1$, $L^2$, $L^\infty$ norms by hand and in code, and explain when each is used in ML.
- Compute the Frobenius norm and spectral norm of a matrix and explain their geometric meaning.
- Identify diagonal, symmetric, orthogonal, PD, and PSD matrices and use their properties.
- Implement Gram-Schmidt orthogonalization from scratch.
- Connect every concept to a concrete ML use case.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup complete.')

---
# Part 1 -- Norms (Section 2.5)

## 1.1 What is a norm?

A **norm** is any function $f : \mathbb{R}^n \to \mathbb{R}$ that satisfies three axioms:

| # | Axiom | Formal statement | Intuition |
|---|-------|------------------|-----------|
| 1 | **Non-negativity & definiteness** | $f(\mathbf{x}) \ge 0$, and $f(\mathbf{x})=0 \iff \mathbf{x}=\mathbf{0}$ | Only the zero vector has zero size. |
| 2 | **Triangle inequality** | $f(\mathbf{x}+\mathbf{y}) \le f(\mathbf{x}) + f(\mathbf{y})$ | Going through a detour is never shorter. |
| 3 | **Absolute homogeneity** | $f(\alpha\mathbf{x}) = |\alpha|\, f(\mathbf{x})$ | Scaling a vector by $\alpha$ scales its size by $|\alpha|$. |

Any function satisfying these three properties gives a valid notion of "size". Different norms measure size differently, which is precisely why we have many norms -- each is useful in different situations.

## 1.2 The $L^p$ norm family

The most common family of vector norms is the **$L^p$ norm** (for $p \ge 1$):

$$\|\mathbf{x}\|_p = \left(\sum_i |x_i|^p\right)^{1/p}$$

Three members dominate ML:

| Norm | $p$ | Formula | Geometric meaning |
|------|-----|---------|--------------------|
| $L^1$ (Manhattan) | 1 | $\sum_i |x_i|$ | Distance walking along axis-aligned streets |
| $L^2$ (Euclidean) | 2 | $\sqrt{\sum_i x_i^2}$ | Straight-line ("as the crow flies") distance |
| $L^\infty$ (max) | $\infty$ | $\max_i |x_i|$ | Size of the single largest component |

### Visualizing unit balls

The **unit ball** of a norm is the set $\{\mathbf{x} : \|\mathbf{x}\|_p \le 1\}$. Its shape reveals the norm's character:

- **$L^1$**: a diamond (square rotated 45 degrees). The corners sit on the coordinate axes.
- **$L^2$**: a circle. All directions treated equally.
- **$L^\infty$**: a square aligned with the axes. Only the largest coordinate matters.

The shape of the $L^1$ ball is the key reason it promotes sparsity in optimization: when you try to find the point on the unit ball closest to some target, the corners (which lie on the axes, meaning some coordinates are exactly zero) are the most likely contact points.

In [ ]:
def lp_unit_ball(p, n_points=500):
    """Return (x, y) coordinates tracing the 2D Lp unit ball boundary."""
    theta = np.linspace(0, 2 * np.pi, n_points)
    x = np.cos(theta)
    y = np.sin(theta)
    # Scale each (cos t, sin t) so its Lp norm equals 1
    if p == np.inf:
        r = np.maximum(np.abs(x), np.abs(y))
    else:
        r = (np.abs(x)**p + np.abs(y)**p) ** (1.0/p)
    return x / r, y / r


fig, axes = plt.subplots(1, 3, figsize=(13, 4))
configs = [
    (1,      '$L^1$ (Manhattan)',  'steelblue',   'diamond -- corners on axes'),
    (2,      '$L^2$ (Euclidean)',  'crimson',     'circle -- isotropic'),
    (np.inf, '$L^\infty$ (max)',   'forestgreen', 'square -- only max coord matters'),
]

for ax, (p, title, color, desc) in zip(axes, configs):
    x, y = lp_unit_ball(p)
    ax.fill(x, y, alpha=0.25, color=color)
    ax.plot(x, y, color=color, linewidth=2)
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.set_title(f'{title}\n({desc})', fontsize=11)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.grid(True, alpha=0.3)

fig.suptitle('Unit Balls for Different $L^p$ Norms in 2D', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Key insight: The L1 ball has corners on the axes.')
print('When an optimization constraint touches the L1 ball, it preferentially')
print('hits a corner -- meaning some coordinates are exactly zero (sparsity).')

### Exercise 1: Implement the $L^p$ Norm from Scratch

Implement `lp_norm(x, p)` for any $p \ge 1$, including $p = \infty$:

$$\|\mathbf{x}\|_p = \begin{cases} \left(\sum_i |x_i|^p\right)^{1/p} & p < \infty \\ \max_i |x_i| & p = \infty \end{cases}$$

**Why this matters:** Every ML loss function, every regularizer, every gradient clipping strategy uses norms. You need to internalize the computation.

In [ ]:
def lp_norm(x, p):
    """
    Compute the Lp norm of vector x.

    Parameters
    ----------
    x : array-like, 1-D
    p : float, p >= 1 or np.inf

    Returns
    -------
    float : the Lp norm
    """
    x = np.asarray(x, dtype=float)
    if p == np.inf:
        return np.max(np.abs(x))
    return np.sum(np.abs(x) ** p) ** (1.0 / p)


# --- Verification ---
x_test = np.array([3.0, -4.0, 0.0, 1.0])

print('Verification against np.linalg.norm:')
for p in [1, 2, 3, np.inf]:
    ours   = lp_norm(x_test, p)
    theirs = np.linalg.norm(x_test, p)
    p_str  = 'inf' if p == np.inf else str(int(p))
    print(f'  p={p_str:<3}: ours={ours:.6f}  numpy={theirs:.6f}')
    assert np.isclose(ours, theirs), f'Mismatch at p={p}'

print('\nAll assertions passed!')

## 1.3 Deep dive: $L^2$ norm (Euclidean norm)

$$\|\mathbf{x}\|_2 = \sqrt{\sum_i x_i^2} = \sqrt{\mathbf{x}^\top \mathbf{x}}$$

This is the "default" norm -- when someone says "the norm" without qualification, they usually mean $L^2$.

**Where it shows up in ML:**
- **MSE loss:** $\frac{1}{n}\|\mathbf{y} - \hat{\mathbf{y}}\|_2^2$ -- the squared $L^2$ norm of the residual vector.
- **$L^2$ regularization (weight decay):** Add $\lambda\|\mathbf{w}\|_2^2$ to the loss. This is equivalent to placing a Gaussian prior $\mathbf{w} \sim \mathcal{N}(0, \frac{1}{2\lambda}I)$ on the weights (Bayesian interpretation).
- **Weight initialization:** Kaiming/He and Xavier/Glorot initialization are designed so that $\|\mathbf{W}\mathbf{x}\|_2 \approx \|\mathbf{x}\|_2$ -- norms are preserved through layers.
- **Gradient clipping:** If $\|\nabla \mathcal{L}\|_2 > \tau$, rescale: $\nabla \mathcal{L} \leftarrow \tau \cdot \frac{\nabla \mathcal{L}}{\|\nabla \mathcal{L}\|_2}$.

### Why we often use $\|\mathbf{x}\|_2^2$ instead of $\|\mathbf{x}\|_2$

The squared $L^2$ norm is more convenient because:
1. It removes the square root, making derivatives cleaner: $\frac{\partial}{\partial x_i}\|\mathbf{x}\|_2^2 = 2x_i$, compared to $\frac{\partial}{\partial x_i}\|\mathbf{x}\|_2 = \frac{x_i}{\|\mathbf{x}\|_2}$ which blows up near $\mathbf{x}=\mathbf{0}$.
2. The gradient $\nabla \|\mathbf{x}\|_2^2 = 2\mathbf{x}$ pushes each component proportionally to its current value.

**Caveat:** $\|\mathbf{x}\|_2^2$ is *not* a norm (it fails absolute homogeneity: $\|\alpha\mathbf{x}\|_2^2 = \alpha^2 \|\mathbf{x}\|_2^2 \ne |\alpha|\|\mathbf{x}\|_2^2$). But as a loss function, it works great.

## 1.4 Deep dive: $L^1$ norm (Manhattan norm) and sparsity

$$\|\mathbf{x}\|_1 = \sum_i |x_i|$$

**The sparsity story (the most important thing to understand):**

When we add $\lambda\|\mathbf{w}\|_1$ to a loss function ("Lasso" regularization), the optimizer tends to push individual weights to *exactly zero*. But why?

**Geometric explanation:** Imagine the loss function has elliptical contours centered away from the origin. The regularization constraint $\|\mathbf{w}\|_1 \le t$ defines a diamond-shaped feasible region. As we shrink $t$, the elliptical contours first touch the diamond at a **corner** -- and the corners of an $L^1$ ball lie on the coordinate axes, where some coordinates are exactly zero.

**Gradient explanation:** The gradient of $|w_i|$ is $\text{sign}(w_i)$ -- a constant push of magnitude 1 toward zero, regardless of how small $w_i$ already is. Compare this with $L^2$ regularization where the gradient $2w_i$ weakens as $w_i \to 0$, never quite reaching it.

**Bayesian explanation:** $L^1$ regularization corresponds to a Laplace prior $p(w_i) \propto e^{-\lambda|w_i|}$, which has a sharp spike at zero. $L^2$ corresponds to a Gaussian prior, which is smooth and rounded at zero.

**When to use $L^1$:** When you believe only a few features matter (feature selection) or when you want an interpretable model with few nonzero weights.

In [ ]:
# Demonstrate the sparsity effect: L1 vs L2 regularization
# Simulate gradient descent with L1 and L2 penalties on a simple quadratic loss

def optimize_with_reg(w_init, loss_grad, reg_type, lam, lr=0.01, steps=500):
    """Run gradient descent with L1 or L2 regularization."""
    w = w_init.copy()
    history = [w.copy()]
    for _ in range(steps):
        grad = loss_grad(w)
        if reg_type == 'L2':
            grad += lam * 2 * w                # gradient of lambda * ||w||^2
        elif reg_type == 'L1':
            grad += lam * np.sign(w)           # subgradient of lambda * ||w||_1
        w = w - lr * grad
        history.append(w.copy())
    return np.array(history)

# Quadratic loss: f(w) = 0.5 * ||w - w*||^2 where w* = [3, 0.05, -0.02, 2, 0.01]
# The last three components are near-zero -- a truly sparse solution would zero them out.
w_star = np.array([3.0, 0.05, -0.02, 2.0, 0.01])
loss_grad = lambda w: (w - w_star)    # gradient of 0.5 * ||w - w*||^2

w0 = np.zeros(5)
lam = 0.5

hist_l1 = optimize_with_reg(w0, loss_grad, 'L1', lam, lr=0.01, steps=1000)
hist_l2 = optimize_with_reg(w0, loss_grad, 'L2', lam, lr=0.01, steps=1000)

print('True signal:   ', w_star)
print(f'L1 final weights: {hist_l1[-1].round(4)}  -- note exact zeros!')
print(f'L2 final weights: {hist_l2[-1].round(4)}  -- all nonzero (just shrunk)')
print(f'\nL1 zero count: {np.sum(np.abs(hist_l1[-1]) < 1e-6)}')
print(f'L2 zero count: {np.sum(np.abs(hist_l2[-1]) < 1e-6)}')

## 1.5 Deep dive: $L^\infty$ norm (max norm)

$$\|\mathbf{x}\|_\infty = \max_i |x_i|$$

This is the limit of $L^p$ as $p \to \infty$ (the largest component dominates). It measures the worst-case magnitude across all dimensions.

**Where it shows up in ML:**
- **Adversarial robustness:** $L^\infty$ perturbation budgets are standard in adversarial ML. An $\epsilon$-ball in $L^\infty$ means each pixel can change by at most $\epsilon$.
- **Max-norm regularization:** Constraining $\|\mathbf{w}\|_\infty$ prevents any single weight from becoming too large, acting as a form of weight clipping.

**Relationship between norms (important inequalities):**

For any $\mathbf{x} \in \mathbb{R}^n$ and $1 \le p \le q \le \infty$:

$$\|\mathbf{x}\|_q \le \|\mathbf{x}\|_p \le n^{1/p - 1/q} \|\mathbf{x}\|_q$$

In particular: $\|\mathbf{x}\|_\infty \le \|\mathbf{x}\|_2 \le \|\mathbf{x}\|_1 \le n \|\mathbf{x}\|_\infty$.

In [ ]:
# Verify the norm inequalities
x = np.array([3.0, -4.0, 0.0, 1.0])
n = len(x)

l1   = lp_norm(x, 1)
l2   = lp_norm(x, 2)
linf = lp_norm(x, np.inf)

print(f'x = {x}')
print(f'||x||_inf = {linf:.4f}')
print(f'||x||_2   = {l2:.4f}')
print(f'||x||_1   = {l1:.4f}')
print(f'n * ||x||_inf = {n * linf:.4f}')
print()
print(f'Check: ||x||_inf <= ||x||_2 <= ||x||_1 <= n * ||x||_inf')
print(f'        {linf:.4f}  <=  {l2:.4f}  <=  {l1:.4f}  <=  {n*linf:.4f}')
assert linf <= l2 + 1e-10 <= l1 + 1e-10 <= n * linf + 1e-10
print('\nInequality chain verified!')

### Exercise 2: Verify the Triangle Inequality

The triangle inequality $\|\mathbf{x}+\mathbf{y}\|_p \le \|\mathbf{x}\|_p + \|\mathbf{y}\|_p$ is what makes norms useful for measuring distance. Let us verify it numerically for random vectors.

In [ ]:
# Verify the triangle inequality holds for many random vectors
np.random.seed(42)
n_trials = 10000
dim = 10

for p in [1, 2, 3, np.inf]:
    violations = 0
    for _ in range(n_trials):
        x = np.random.randn(dim)
        y = np.random.randn(dim)
        lhs = lp_norm(x + y, p)
        rhs = lp_norm(x, p) + lp_norm(y, p)
        if lhs > rhs + 1e-10:  # small tolerance for floating point
            violations += 1
    p_str = 'inf' if p == np.inf else str(int(p))
    print(f'p={p_str:<3}: {violations} violations out of {n_trials} trials')

print('\nTriangle inequality holds for all tested cases (as it must, by theorem).')

## 1.6 Matrix norms: Frobenius and Spectral

We often need to measure the "size" of a matrix. Two norms dominate ML:

### Frobenius norm

$$\|A\|_F = \sqrt{\sum_{i,j} A_{i,j}^2} = \sqrt{\text{tr}(A^\top A)} = \sqrt{\sum_i \sigma_i^2}$$

The Frobenius norm treats the matrix as a flattened vector and computes the $L^2$ norm. The three equivalent formulations are:
1. **Element-wise:** Sum of squared entries, then square root.
2. **Via trace:** $\text{tr}(A^\top A) = \sum_i [A^\top A]_{ii} = \sum_i \sum_j A_{ji}^2 = \sum_{i,j} A_{i,j}^2$.
3. **Via singular values:** Since the singular values capture the "energy" of $A$, $\|A\|_F^2 = \sum \sigma_i^2$.

### Spectral norm (operator norm)

$$\|A\|_2 = \sigma_{\max}(A) = \max_{\|\mathbf{x}\|_2 = 1} \|A\mathbf{x}\|_2$$

The spectral norm is the **largest singular value** of $A$. Geometrically, it is the maximum factor by which $A$ can stretch a unit vector. This is the **Lipschitz constant** of the linear map $\mathbf{x} \mapsto A\mathbf{x}$.

**ML connections:**
- **Spectral normalization (GANs):** Divide each weight matrix by its spectral norm to enforce Lipschitz continuity of the discriminator: $W \leftarrow W / \sigma_{\max}(W)$. This stabilizes GAN training.
- **Conditioning:** The ratio $\sigma_{\max} / \sigma_{\min}$ (condition number) determines how sensitive a linear system is to perturbations. Ill-conditioned matrices make optimization harder.

### Exercise 3: Frobenius Norm -- Three Equivalent Computations

Implement three functions, each computing the Frobenius norm using a different formulation, and verify they agree.

In [ ]:
def frobenius_norm(A):
    """Frobenius norm via direct element-wise computation."""
    A = np.asarray(A, dtype=float)
    return np.sqrt(np.sum(A ** 2))


def frobenius_via_trace(A):
    """Frobenius norm via sqrt(trace(A^T A))."""
    A = np.asarray(A, dtype=float)
    return np.sqrt(np.trace(A.T @ A))


def frobenius_via_svd(A):
    """Frobenius norm via sqrt(sum of squared singular values)."""
    A = np.asarray(A, dtype=float)
    sigma = np.linalg.svd(A, compute_uv=False)
    return np.sqrt(np.sum(sigma ** 2))


# --- Verification ---
A_test = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]], dtype=float)

ref = np.linalg.norm(A_test, 'fro')
f1  = frobenius_norm(A_test)
f2  = frobenius_via_trace(A_test)
f3  = frobenius_via_svd(A_test)

print(f'numpy  reference : {ref:.8f}')
print(f'direct elements  : {f1:.8f}')
print(f'via trace(A^T A) : {f2:.8f}')
print(f'via SVD sigmas   : {f3:.8f}')

assert np.isclose(f1, ref), 'frobenius_norm mismatch'
assert np.isclose(f2, ref), 'frobenius_via_trace mismatch'
assert np.isclose(f3, ref), 'frobenius_via_svd mismatch'
print('\nAll three methods agree!')

### Exercise 4: Spectral Norm and Stretching

The spectral norm $\|A\|_2 = \sigma_{\max}(A)$ tells us the maximum factor by which $A$ stretches any unit vector. Let us compute it and visualize the stretching.

In [ ]:
def max_norm(A):
    """Max norm: max over all elements of |A_ij|."""
    A = np.asarray(A, dtype=float)
    return np.max(np.abs(A))


def spectral_norm(A):
    """Spectral norm: largest singular value of A."""
    A = np.asarray(A, dtype=float)
    return np.max(np.linalg.svd(A, compute_uv=False))


# --- Compare norms on several matrices ---
matrices = {
    'Identity (3x3)':       np.eye(3),
    'Scaling (diag 1,2,5)': np.diag([1., 2., 5.]),
    'Random (4x4)':         np.random.randn(4, 4),
    'Rank-1 outer product': np.outer([1,2,3,4], [1,1,1,1]).astype(float),
}

print(f'{"Matrix":<28}  {"Max norm":>10}  {"Spectral norm":>14}  {"Frobenius":>12}')
print('-' * 70)
for name, M in matrices.items():
    mn = max_norm(M)
    sn = spectral_norm(M)
    fn = np.linalg.norm(M, 'fro')
    print(f'{name:<28}  {mn:>10.4f}  {sn:>14.4f}  {fn:>12.4f}')

# Key relationship: spectral_norm <= frobenius_norm (always)
print('\nNote: spectral norm <= Frobenius norm always (equality iff rank 1).')
print('For the rank-1 matrix, they are equal because there is only one nonzero singular value.')

for name, M in matrices.items():
    sn = spectral_norm(M)
    fn = np.linalg.norm(M, 'fro')
    assert sn <= fn + 1e-10, f'Spectral > Frobenius for {name}!'
print('\nAssertion passed: spectral <= Frobenius for all test matrices.')

In [ ]:
# Visualize how a matrix transforms the unit circle
# The spectral norm is the radius of the largest stretching direction

A = np.array([[2.0, 1.0],
              [0.5, 1.5]])

U, S, Vt = np.linalg.svd(A)
print(f'Singular values: {S}')
print(f'Spectral norm (sigma_max) = {S[0]:.4f}')
print(f'This means A can stretch a unit vector by at most a factor of {S[0]:.4f}')

theta = np.linspace(0, 2*np.pi, 200)
circle = np.vstack([np.cos(theta), np.sin(theta)])  # 2 x 200
ellipse = A @ circle  # transformed

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(circle[0], circle[1], 'steelblue', linewidth=2)
ax1.set_xlim(-3, 3); ax1.set_ylim(-3, 3)
ax1.set_aspect('equal')
ax1.set_title('Unit circle (input)', fontsize=12)
ax1.axhline(0, color='k', linewidth=0.5)
ax1.axvline(0, color='k', linewidth=0.5)
ax1.grid(True, alpha=0.3)

ax2.plot(ellipse[0], ellipse[1], 'crimson', linewidth=2)
# Draw the direction of maximum stretching
v_max = Vt[0]  # right singular vector (input direction)
u_max = U[:, 0]  # left singular vector (output direction)
ax2.annotate('', xy=S[0]*u_max, xytext=[0,0],
             arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
ax2.text(S[0]*u_max[0]*1.1, S[0]*u_max[1]*1.1,
         f'$\\sigma_{{max}}={S[0]:.2f}$', fontsize=12, color='green', fontweight='bold')
ax2.set_xlim(-3, 3); ax2.set_ylim(-3, 3)
ax2.set_aspect('equal')
ax2.set_title('After $A\\mathbf{x}$ (output) -- an ellipse', fontsize=12)
ax2.axhline(0, color='k', linewidth=0.5)
ax2.axvline(0, color='k', linewidth=0.5)
ax2.grid(True, alpha=0.3)

fig.suptitle('Spectral Norm = Maximum Stretching Factor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 5: Norms in ML -- L2 Regularization as Weight Decay

Let us verify a key fact: adding $\lambda\|\mathbf{w}\|_2^2$ to the loss is equivalent to multiplying the weights by $(1 - 2\lambda\eta)$ at each gradient step, where $\eta$ is the learning rate. This is why $L^2$ regularization is called **weight decay**.

$$\mathbf{w}_{t+1} = \mathbf{w}_t - \eta\left(\nabla\mathcal{L} + 2\lambda\mathbf{w}_t\right) = (1 - 2\lambda\eta)\mathbf{w}_t - \eta\nabla\mathcal{L}$$

The factor $(1 - 2\lambda\eta)$ shrinks all weights toward zero every step -- that is the "decay".

In [ ]:
# Demonstrate weight decay equivalence
np.random.seed(7)

# Setup: simple linear regression y = Xw + noise
n, d = 50, 5
X = np.random.randn(n, d)
w_true = np.array([3.0, -1.0, 0.0, 2.0, 0.0])
y = X @ w_true + 0.1 * np.random.randn(n)

lr = 0.001
lam = 0.1
steps = 500

# Method 1: Explicit L2 regularization in loss gradient
w1 = np.zeros(d)
for t in range(steps):
    grad_loss = (2.0 / n) * X.T @ (X @ w1 - y)  # gradient of MSE
    grad_reg  = 2 * lam * w1                       # gradient of lambda * ||w||^2
    w1 = w1 - lr * (grad_loss + grad_reg)

# Method 2: Weight decay (multiply by decay factor, then subtract loss gradient)
w2 = np.zeros(d)
decay_factor = 1 - 2 * lam * lr
for t in range(steps):
    grad_loss = (2.0 / n) * X.T @ (X @ w2 - y)
    w2 = decay_factor * w2 - lr * grad_loss

print('True weights:         ', w_true)
print('L2 reg final weights: ', w1.round(4))
print('Weight decay final:   ', w2.round(4))
print(f'Max difference:        {np.max(np.abs(w1 - w2)):.2e}')
assert np.allclose(w1, w2, atol=1e-10), 'Methods should produce identical results'
print('\nPerfect agreement -- L2 regularization IS weight decay.')

---
# Part 2 -- Special Matrices (Section 2.6)

Certain matrices have special structure that makes computation faster and theory cleaner. The Goodfellow text highlights several; each shows up constantly in ML.

## 2.1 Diagonal matrices

A matrix $D$ is **diagonal** if $D_{i,j} = 0$ for all $i \ne j$. Only the diagonal entries can be nonzero.

**Why they matter:**
- **Efficient multiplication:** $D\mathbf{x}$ is just element-wise multiplication: $(D\mathbf{x})_i = d_i x_i$. This is $O(n)$ instead of $O(n^2)$.
- **Trivial inverse:** $D^{-1} = \text{diag}(1/d_1, \ldots, 1/d_n)$ (when all $d_i \ne 0$). Just reciprocate each diagonal entry.
- **Eigenvalues are the diagonal entries:** The eigenvectors are the standard basis vectors.

**ML connection:** Diagonal approximations to the Hessian or Fisher information matrix are used in optimizers like Adam (which maintains per-parameter adaptive learning rates -- essentially a diagonal preconditioner).

In [ ]:
# Diagonal matrices: efficient computation
d = np.array([2.0, 5.0, -3.0, 1.0])
D = np.diag(d)
x = np.array([1.0, 2.0, 3.0, 4.0])

print('D =')
print(D)
print(f'\nx = {x}')
print(f'\nD @ x (matrix multiply):    {D @ x}')
print(f'd * x (element-wise):        {d * x}')
assert np.allclose(D @ x, d * x)
print('Same result! Element-wise multiply is O(n) vs O(n^2) for full matrix.')

# Inverse
D_inv = np.diag(1.0 / d)
print(f'\nD^{{-1}} @ D (should be I):')
print(np.round(D_inv @ D, 10))
assert np.allclose(D_inv @ D, np.eye(4))

# Timing comparison
n = 5000
d_big = np.random.randn(n)
D_big = np.diag(d_big)
x_big = np.random.randn(n)

t0 = time.time()
for _ in range(100):
    _ = D_big @ x_big
t_mat = time.time() - t0

t0 = time.time()
for _ in range(100):
    _ = d_big * x_big
t_elem = time.time() - t0

print(f'\nTiming (n={n}, 100 repeats):')
print(f'  Full matrix multiply: {t_mat:.4f}s')
print(f'  Element-wise:         {t_elem:.4f}s')
print(f'  Speedup:              {t_mat/t_elem:.1f}x')

## 2.2 Symmetric matrices

A matrix $A$ is **symmetric** if $A = A^\top$ (i.e., $A_{i,j} = A_{j,i}$).

**Why they matter (the Spectral Theorem):**

Real symmetric matrices have the spectral decomposition:
$$A = Q \Lambda Q^\top$$
where $Q$ is orthogonal (columns are the eigenvectors) and $\Lambda$ is diagonal (eigenvalues on the diagonal). This means:
1. **All eigenvalues are real** (not complex).
2. **Eigenvectors are orthogonal** to each other.
3. **The matrix is fully diagonalizable.**

This is the foundation of Section 2.7 (eigendecomposition) and appears constantly in ML.

**ML connections:**
- **Covariance matrices** are always symmetric (and PSD): $\Sigma = \frac{1}{n}X^\top X$.
- **The Hessian** $\nabla^2 \mathcal{L}$ of any twice-differentiable loss is symmetric (mixed partials commute).
- **Kernel matrices** (Gram matrices) $K_{ij} = k(x_i, x_j)$ are symmetric.

**Any square matrix can be decomposed** into symmetric + anti-symmetric parts:
$$A = \underbrace{\frac{A + A^\top}{2}}_{\text{symmetric}} + \underbrace{\frac{A - A^\top}{2}}_{\text{anti-symmetric}}$$

### Exercise 6: Symmetric Decomposition

Implement functions to check symmetry, extract the symmetric part, and extract the anti-symmetric part of a matrix.

In [ ]:
def is_symmetric(A, tol=1e-10):
    """Return True if A is symmetric (A == A^T) within tolerance."""
    A = np.asarray(A, dtype=float)
    return np.allclose(A, A.T, atol=tol)


def symmetric_part(A):
    """Return the symmetric part of A: (A + A^T) / 2."""
    A = np.asarray(A, dtype=float)
    return (A + A.T) / 2


def antisymmetric_part(A):
    """Return the anti-symmetric part of A: (A - A^T) / 2."""
    A = np.asarray(A, dtype=float)
    return (A - A.T) / 2


# --- Verification ---
A_test = np.array([[4, 2, 7],
                   [1, 5, 3],
                   [8, 6, 9]], dtype=float)

S = symmetric_part(A_test)
K = antisymmetric_part(A_test)

print('Original A:')
print(A_test)
print('\nSymmetric part S = (A + A^T) / 2:')
print(S)
print('\nAnti-symmetric part K = (A - A^T) / 2:')
print(K)
print('\nS + K (should equal A):')
print(S + K)

assert is_symmetric(S),               'S should be symmetric'
assert np.allclose(K, -K.T),          'K should be anti-symmetric (K = -K^T)'
assert np.allclose(S + K, A_test),    'S + K should reconstruct A'
assert not is_symmetric(A_test),      'A_test is not symmetric'
assert is_symmetric(A_test + A_test.T), 'A + A^T should be symmetric'
print('\nAll assertions passed!')

## 2.3 Orthogonal matrices

A square matrix $Q$ is **orthogonal** if:
$$Q^\top Q = Q Q^\top = I \quad \Longleftrightarrow \quad Q^{-1} = Q^\top$$

This means the columns (and rows) of $Q$ form an **orthonormal** set: they are mutually perpendicular and each has unit length.

**Key property -- norm preservation:** For any vector $\mathbf{x}$:
$$\|Q\mathbf{x}\|_2^2 = (Q\mathbf{x})^\top(Q\mathbf{x}) = \mathbf{x}^\top Q^\top Q \mathbf{x} = \mathbf{x}^\top I \mathbf{x} = \|\mathbf{x}\|_2^2$$

So $\|Q\mathbf{x}\|_2 = \|\mathbf{x}\|_2$ -- orthogonal matrices **preserve lengths** (and angles). Geometrically, they are **rotations** (if $\det Q = +1$) or **reflections** (if $\det Q = -1$).

**Why this matters in ML:**
- **Orthogonal initialization:** If weight matrices are orthogonal, forward and backward passes preserve gradient norms. This prevents vanishing/exploding gradients in deep networks.
- **Cheap inverse:** $Q^{-1} = Q^\top$ is free -- no need to solve a linear system.
- **Numerical stability:** Orthogonal transformations do not amplify rounding errors.
- **QR decomposition:** Used in many optimization algorithms and for solving least squares.

In [ ]:
# Build a 2D rotation matrix and verify properties
def rotation_2d(theta_deg):
    """Return 2x2 rotation matrix for angle theta (in degrees)."""
    theta = np.radians(theta_deg)
    return np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta),  np.cos(theta)]])


Q = rotation_2d(37)
print(f'2D rotation by 37 degrees:\n{Q}\n')

# Verify orthogonality
print('Q^T Q (should be I):')
print(np.round(Q.T @ Q, 10))
assert np.allclose(Q.T @ Q, np.eye(2))
print(f'det(Q) = {np.linalg.det(Q):.6f}  (should be +1 for proper rotation)')

# Verify norm preservation
v = np.array([3.0, 4.0])
Qv = Q @ v
print(f'\n||v||_2   = {np.linalg.norm(v):.6f}')
print(f'||Qv||_2  = {np.linalg.norm(Qv):.6f}  (preserved!)')
assert np.isclose(np.linalg.norm(v), np.linalg.norm(Qv))

# Visualize rotation
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
theta_vis = np.linspace(0, 2*np.pi, 200)
circle = np.vstack([np.cos(theta_vis), np.sin(theta_vis)])
rotated_circle = Q @ circle

ax.plot(circle[0], circle[1], 'steelblue', linewidth=1, alpha=0.5, label='Unit circle')
ax.plot(rotated_circle[0], rotated_circle[1], 'crimson', linewidth=1, alpha=0.5,
        linestyle='--', label='Rotated unit circle (same!)')

# Show v and Qv
ax.annotate('', xy=v, xytext=[0,0],
            arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.5))
ax.text(v[0]+0.1, v[1]+0.1, f'$v = ({v[0]:.0f}, {v[1]:.0f})$',
        fontsize=11, color='steelblue')
ax.annotate('', xy=Qv, xytext=[0,0],
            arrowprops=dict(arrowstyle='->', color='crimson', lw=2.5))
ax.text(Qv[0]+0.1, Qv[1]+0.1, f'$Qv = ({Qv[0]:.1f}, {Qv[1]:.1f})$',
        fontsize=11, color='crimson')

ax.set_xlim(-6, 6); ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
ax.set_title('Orthogonal matrix = rotation (preserves length)', fontsize=13)
plt.tight_layout()
plt.show()

## 2.4 Positive definite (PD) and positive semidefinite (PSD) matrices

For a symmetric matrix $A$:

| Property | Condition | Eigenvalues | ML Example |
|----------|-----------|-------------|------------|
| **Positive definite (PD)** | $\mathbf{x}^\top A \mathbf{x} > 0$ for all $\mathbf{x} \ne \mathbf{0}$ | All $\lambda_i > 0$ | Hessian at a strict local minimum |
| **Positive semidefinite (PSD)** | $\mathbf{x}^\top A \mathbf{x} \ge 0$ for all $\mathbf{x}$ | All $\lambda_i \ge 0$ | Covariance matrices, Gram matrices |
| **Indefinite** | $\mathbf{x}^\top A \mathbf{x}$ can be positive or negative | Mixed signs | Hessian at a saddle point |

**Why PD/PSD matters in ML:**

1. **Optimization:** The Hessian $\nabla^2 \mathcal{L}$ being PD at a point means that point is a strict local minimum. If it is indefinite, it is a saddle point. This is central to understanding when gradient descent converges.

2. **Covariance matrices:** Any covariance matrix $\Sigma = \mathbb{E}[(\mathbf{x}-\boldsymbol{\mu})(\mathbf{x}-\boldsymbol{\mu})^\top]$ is PSD because:
$$\mathbf{v}^\top \Sigma \mathbf{v} = \mathbb{E}[(\mathbf{v}^\top(\mathbf{x}-\boldsymbol{\mu}))^2] \ge 0$$
   A variance (even after projection) can never be negative.

3. **Kernel methods:** A valid kernel function must produce PSD Gram matrices.

**Three equivalent tests for PD:**
1. **Eigenvalue test:** All eigenvalues $\lambda_i > 0$.
2. **Cholesky test:** Cholesky decomposition $A = LL^\top$ succeeds.
3. **Sylvester's criterion:** All leading principal minors are positive ($\det(A_{1:k,1:k}) > 0$ for $k = 1,\ldots,n$).

### Exercise 7: Three Tests for Positive Definiteness

Implement all three PD tests and verify they agree on several test matrices.

In [ ]:
def is_pd_eigenvalue(A):
    """Test PD via eigenvalues: True iff all eigenvalues > 0."""
    A = np.asarray(A, dtype=float)
    eigenvalues = np.linalg.eigvalsh(A)  # eigvalsh for symmetric matrices
    return bool(np.all(eigenvalues > 0))


def is_pd_cholesky(A):
    """Test PD via Cholesky: True iff Cholesky decomposition succeeds."""
    A = np.asarray(A, dtype=float)
    try:
        np.linalg.cholesky(A)
        return True
    except np.linalg.LinAlgError:
        return False


def is_pd_sylvester(A):
    """Test PD via Sylvester's criterion: all leading principal minors > 0."""
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    for k in range(1, n + 1):
        if np.linalg.det(A[:k, :k]) <= 0:
            return False
    return True


# --- Test matrices ---
# 1. PD: A^T A + eps*I guarantees PD
B = np.random.randn(4, 4)
A_pd  = B.T @ B + 3 * np.eye(4)

# 2. PSD but not PD: rank-1 outer product has eigenvalues 0,0,0,||v||^2
v = np.array([1., 2., 3., 4.])
A_psd = np.outer(v, v)

# 3. Indefinite: has both positive and negative eigenvalues
A_indef = np.array([[2., 3.],
                    [3., 1.]])

print(f'{"Matrix":<18}  {"Eigenvalue":>12}  {"Cholesky":>10}  {"Sylvester":>10}')
print('-' * 57)
for name, M in [('PD', A_pd), ('PSD (rank-1)', A_psd), ('Indefinite', A_indef)]:
    e = is_pd_eigenvalue(M)
    c = is_pd_cholesky(M)
    s = is_pd_sylvester(M)
    print(f'{name:<18}  {str(e):>12}  {str(c):>10}  {str(s):>10}')

# Verify PD matrix passes all three
assert is_pd_eigenvalue(A_pd) and is_pd_cholesky(A_pd) and is_pd_sylvester(A_pd), \
    'All three should return True for PD matrix'
# PSD and indefinite fail
assert not is_pd_eigenvalue(A_psd),   'PSD (not PD) should fail eigenvalue test'
assert not is_pd_eigenvalue(A_indef), 'Indefinite should fail eigenvalue test'
print('\nAll assertions passed!')

In [ ]:
# Visualize PD vs indefinite via the quadratic form x^T A x
# PD -> bowl shape (minimum exists), Indefinite -> saddle shape

A_pd_2d    = np.array([[2.0, 0.5],
                       [0.5, 3.0]])   # eigenvalues: 1.76, 3.24 -- both positive
A_indef_2d = np.array([[2.0, 3.0],
                       [3.0, 1.0]])   # eigenvalues: -1.61, 4.61 -- mixed

fig = plt.figure(figsize=(14, 5))

for idx, (A_mat, title) in enumerate([(A_pd_2d, 'Positive Definite (bowl = minimum)'),
                                       (A_indef_2d, 'Indefinite (saddle point)')]):
    ax = fig.add_subplot(1, 2, idx + 1, projection='3d')
    grid = np.linspace(-2, 2, 80)
    X1, X2 = np.meshgrid(grid, grid)
    Z = np.zeros_like(X1)
    for i in range(len(grid)):
        for j in range(len(grid)):
            xv = np.array([X1[i,j], X2[i,j]])
            Z[i,j] = xv @ A_mat @ xv

    ax.plot_surface(X1, X2, Z, cmap='coolwarm', alpha=0.8, edgecolor='none')
    eigs = np.linalg.eigvalsh(A_mat)
    ax.set_title(f'{title}\n$\\lambda$ = {eigs.round(2)}', fontsize=11)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$'); ax.set_zlabel('$x^T A x$')

plt.tight_layout()
plt.show()
print('Left: PD matrix -> the quadratic form is a bowl. The origin is a minimum.')
print('Right: Indefinite -> a saddle. The origin is NOT a minimum.')
print('This is exactly how the Hessian determines the nature of critical points in optimization.')

### Why covariance matrices are always PSD

This is worth proving because covariance matrices appear everywhere in ML (PCA, Gaussian distributions, kernel methods).

Let $X$ be a data matrix with rows being centered data points. The covariance matrix is:
$$\Sigma = \frac{1}{n} X^\top X$$

For any vector $\mathbf{v}$:
$$\mathbf{v}^\top \Sigma \mathbf{v} = \frac{1}{n} \mathbf{v}^\top X^\top X \mathbf{v} = \frac{1}{n} \|X\mathbf{v}\|_2^2 \ge 0$$

The squared norm of any vector is non-negative. So $\Sigma$ is PSD.

In [ ]:
# Verify: covariance matrices are always PSD
np.random.seed(99)
n, d = 100, 5
X = np.random.randn(n, d)
X = X - X.mean(axis=0)  # center

Sigma = (X.T @ X) / n
eigenvalues = np.linalg.eigvalsh(Sigma)

print(f'Covariance matrix shape: {Sigma.shape}')
print(f'Eigenvalues: {eigenvalues.round(4)}')
print(f'All eigenvalues >= 0? {np.all(eigenvalues >= -1e-10)}')
print(f'Is symmetric? {is_symmetric(Sigma)}')

# Test with random projection vectors
for _ in range(1000):
    v = np.random.randn(d)
    quadratic_form = v @ Sigma @ v
    assert quadratic_form >= -1e-10, f'Found negative quadratic form: {quadratic_form}'
print('\nv^T Sigma v >= 0 verified for 1000 random vectors. Covariance is PSD.')

## 2.5 Gram-Schmidt orthogonalization

Given linearly independent vectors $\{\mathbf{v}_1, \ldots, \mathbf{v}_k\}$, the **Gram-Schmidt process** produces an orthonormal basis $\{\mathbf{q}_1, \ldots, \mathbf{q}_k\}$ spanning the same subspace:

$$\mathbf{u}_j = \mathbf{v}_j - \sum_{i=1}^{j-1} \langle \mathbf{v}_j, \mathbf{q}_i \rangle \, \mathbf{q}_i, \qquad \mathbf{q}_j = \frac{\mathbf{u}_j}{\|\mathbf{u}_j\|}$$

**Intuition:** For each new vector, subtract its projection onto all previously computed orthonormal vectors. What remains is the component perpendicular to the existing subspace. Normalize it.

This gives the **thin QR decomposition**: $V = QR$ where $Q$ has orthonormal columns and $R$ is upper triangular.

**ML connection:** Gram-Schmidt and QR decomposition are used in solving least squares problems, in Lanczos/Arnoldi iterations for eigenvalue computation, and in the orthogonal Procrustes problem (aligning word embeddings across languages).

### Exercise 8: Implement Gram-Schmidt from Scratch

In [ ]:
def gram_schmidt(V):
    """
    Classical Gram-Schmidt orthogonalization.

    Parameters
    ----------
    V : 2-D array of shape (n, k) whose columns are linearly independent

    Returns
    -------
    Q : (n, k) array with orthonormal columns
    R : (k, k) upper-triangular array such that V = Q @ R
    """
    V = np.asarray(V, dtype=float)
    n, k = V.shape
    Q = np.zeros((n, k))
    R = np.zeros((k, k))

    for j in range(k):
        # Start with the original vector
        u = V[:, j].copy()

        # Subtract projections onto all previous q vectors
        for i in range(j):
            R[i, j] = Q[:, i] @ V[:, j]  # projection coefficient
            u -= R[i, j] * Q[:, i]

        # Normalize
        R[j, j] = np.linalg.norm(u)
        Q[:, j] = u / R[j, j]

    return Q, R


# --- Verification ---
np.random.seed(42)
V = np.random.randn(5, 3)  # 5D space, 3 vectors
Q, R = gram_schmidt(V)

print('Q^T Q (should be identity):')
print(np.round(Q.T @ Q, 10))
assert np.allclose(Q.T @ Q, np.eye(3), atol=1e-10), 'Q columns not orthonormal'
print('Q^T Q = I verified.\n')

# Check Q @ R == V
print('||V - Q R||_F (should be ~0):', np.linalg.norm(V - Q @ R, 'fro'))
assert np.allclose(V, Q @ R, atol=1e-10), 'QR decomposition failed'
print('V = QR verified.\n')

# Compare with numpy QR
Q_np, R_np = np.linalg.qr(V)
# Columns may differ by sign
for j in range(3):
    sign = np.sign(Q[0, j]) * np.sign(Q_np[0, j])
    assert np.allclose(Q[:, j], sign * Q_np[:, j], atol=1e-10), f'Column {j} mismatch'
print('Columns match numpy QR (up to sign). All assertions passed!')

In [ ]:
# Visualize Gram-Schmidt in 2D: step by step

v1 = np.array([2.0, 1.0])
v2 = np.array([1.0, 2.0])

# Step 1: q1 = v1 / ||v1||
q1 = v1 / np.linalg.norm(v1)

# Step 2: u2 = v2 - <v2, q1> q1, then q2 = u2 / ||u2||
proj = np.dot(v2, q1) * q1
u2 = v2 - proj
q2 = u2 / np.linalg.norm(u2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax in axes:
    ax.set_xlim(-0.5, 3)
    ax.set_ylim(-1, 3)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)

# Panel 1: Original vectors
axes[0].annotate('', xy=v1, xytext=[0,0],
                 arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.5))
axes[0].annotate('', xy=v2, xytext=[0,0],
                 arrowprops=dict(arrowstyle='->', color='crimson', lw=2.5))
axes[0].text(v1[0]+0.05, v1[1]+0.1, '$v_1$', fontsize=13, color='steelblue')
axes[0].text(v2[0]+0.05, v2[1]+0.1, '$v_2$', fontsize=13, color='crimson')
axes[0].set_title('Step 0: Original vectors', fontsize=11)

# Panel 2: Normalize v1 -> q1, show projection of v2
axes[1].annotate('', xy=q1, xytext=[0,0],
                 arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.5))
axes[1].annotate('', xy=v2, xytext=[0,0],
                 arrowprops=dict(arrowstyle='->', color='crimson', lw=2.5))
axes[1].annotate('', xy=proj, xytext=[0,0],
                 arrowprops=dict(arrowstyle='->', color='orange', lw=2, linestyle='dashed'))
axes[1].annotate('', xy=v2, xytext=proj,
                 arrowprops=dict(arrowstyle='->', color='green', lw=2))
axes[1].text(q1[0]+0.05, q1[1]+0.1, '$q_1$', fontsize=13, color='steelblue')
axes[1].text(v2[0]+0.05, v2[1]+0.1, '$v_2$', fontsize=13, color='crimson')
axes[1].text(proj[0]+0.05, proj[1]-0.25, 'proj', fontsize=10, color='orange')
axes[1].text((proj[0]+v2[0])/2+0.1, (proj[1]+v2[1])/2, '$u_2$', fontsize=11, color='green')
axes[1].set_title('Step 1: Subtract projection', fontsize=11)

# Panel 3: Final orthonormal basis
axes[2].annotate('', xy=q1, xytext=[0,0],
                 arrowprops=dict(arrowstyle='->', color='steelblue', lw=2.5))
axes[2].annotate('', xy=q2, xytext=[0,0],
                 arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
axes[2].text(q1[0]+0.05, q1[1]+0.1, '$q_1$', fontsize=13, color='steelblue')
axes[2].text(q2[0]+0.05, q2[1]+0.1, '$q_2$', fontsize=13, color='green')
axes[2].set_title('Step 2: Orthonormal basis', fontsize=11)

fig.suptitle('Gram-Schmidt Process: Step by Step', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'q1 . q2 = {np.dot(q1, q2):.10f}  (orthogonal!)')
print(f'||q1|| = {np.linalg.norm(q1):.6f}, ||q2|| = {np.linalg.norm(q2):.6f}  (unit length!)')

## 2.6 Putting it all together: Rodrigues' rotation formula

As a capstone exercise, let us implement **Rodrigues' rotation formula**, which constructs a 3D rotation matrix from an axis $\hat{\mathbf{k}}$ and angle $\theta$:

$$R = I \cos\theta + \sin\theta\, [\hat{\mathbf{k}}]_{\times} + (1 - \cos\theta)\, \hat{\mathbf{k}}\hat{\mathbf{k}}^\top$$

where $[\hat{\mathbf{k}}]_{\times}$ is the skew-symmetric matrix associated with the cross product.

This exercise combines several concepts:
- The result $R$ is **orthogonal** (rotation preserves norms).
- The intermediate matrix $[\hat{\mathbf{k}}]_{\times}$ is **anti-symmetric** ($K = -K^\top$).
- The outer product $\hat{\mathbf{k}}\hat{\mathbf{k}}^\top$ is **symmetric** and **PSD** (rank 1).

### Exercise 9: Rodrigues' Rotation

In [ ]:
def skew_symmetric(v):
    """Build the 3x3 skew-symmetric matrix [v]_x such that [v]_x @ w = v x w."""
    v = np.asarray(v, dtype=float)
    return np.array([[    0, -v[2],  v[1]],
                     [ v[2],     0, -v[0]],
                     [-v[1],  v[0],     0]])


def rodrigues_rotation(axis, theta_deg):
    """
    Build 3x3 rotation matrix using Rodrigues' formula.

    Parameters
    ----------
    axis      : array of shape (3,), rotation axis (need not be unit)
    theta_deg : float, rotation angle in degrees

    Returns
    -------
    R : 3x3 rotation matrix (orthogonal, det=+1)
    """
    axis = np.asarray(axis, dtype=float)
    axis = axis / np.linalg.norm(axis)        # normalize to unit vector
    K = skew_symmetric(axis)                   # skew-symmetric
    theta = np.radians(theta_deg)
    # Rodrigues' formula
    R = np.eye(3) * np.cos(theta) + np.sin(theta) * K + (1 - np.cos(theta)) * np.outer(axis, axis)
    return R


# --- Verification ---
# Rotation by 90 degrees around z-axis: maps x -> y
R_z90 = rodrigues_rotation([0, 0, 1], 90)
x_hat = np.array([1., 0., 0.])
y_hat = np.array([0., 1., 0.])
print('R_z(90) @ [1,0,0] (should be [0,1,0]):', np.round(R_z90 @ x_hat, 8))
assert np.allclose(R_z90 @ x_hat, y_hat, atol=1e-10), 'z-axis rotation failed'

# Check orthogonality and det=+1 for various angles
axis_test = [1., 2., 3.]
for angle in [0, 30, 90, 137, 270]:
    R = rodrigues_rotation(axis_test, angle)
    assert np.allclose(R.T @ R, np.eye(3), atol=1e-10), f'Not orthogonal at {angle} deg'
    assert np.isclose(np.linalg.det(R), 1.0, atol=1e-10), f'det != 1 at {angle} deg'
print('Orthogonality and det=+1 verified for all test angles.')

# Verify norm preservation
v_test = np.array([1., 2., 3.])
R60 = rodrigues_rotation([1, 1, 0], 60)
print(f'||v|| = {np.linalg.norm(v_test):.6f}')
print(f'||Rv|| = {np.linalg.norm(R60 @ v_test):.6f}  (preserved!)')
assert np.isclose(np.linalg.norm(v_test), np.linalg.norm(R60 @ v_test))

In [ ]:
# Visualize 3D rotation of a point cloud
np.random.seed(3)

# Create an arrow-shaped point cloud
t_line = np.linspace(0, 1, 30)
shaft  = np.column_stack([t_line, np.zeros(30), np.zeros(30)])
tip_u  = np.column_stack([0.8 + 0.2*t_line,  0.2*t_line, np.zeros(30)])
tip_d  = np.column_stack([0.8 + 0.2*t_line, -0.2*t_line, np.zeros(30)])
cloud  = np.vstack([shaft, tip_u, tip_d])

R_final = rodrigues_rotation([0, 0, 1], 60)
cloud_rotated = (R_final @ cloud.T).T

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(cloud[:, 0], cloud[:, 1], cloud[:, 2],
           c='steelblue', s=20, label='Original', alpha=0.8)
ax.scatter(cloud_rotated[:, 0], cloud_rotated[:, 1], cloud_rotated[:, 2],
           c='crimson', s=20, label='Rotated 60 deg around z', alpha=0.8)

# Draw rotation axis
ax.quiver(0, 0, -0.4, 0, 0, 0.8, color='green', linewidth=3,
          arrow_length_ratio=0.2, label='Rotation axis (z)')

ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title("Rodrigues' Rotation: 3D Arrow Rotated 60 deg around Z", fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()
print(f'Rotation matrix:\n{np.round(R_final, 4)}')

---
# Summary

## Part 1: Norms (Section 2.5)

| Norm | Formula | ML Use Case |
|------|---------|-------------|
| $L^1$ | $\sum_i |x_i|$ | Lasso / sparsity / Laplace prior / feature selection |
| $L^2$ | $\sqrt{\sum_i x_i^2}$ | Weight decay / Gaussian prior / MSE / gradient clipping |
| $L^\infty$ | $\max_i |x_i|$ | Adversarial robustness / weight clipping |
| Frobenius | $\sqrt{\sum_{ij} A_{ij}^2}$ | Matrix regularization / low-rank approximation |
| Spectral | $\sigma_{\max}(A)$ | Spectral normalization (GANs) / Lipschitz constant / conditioning |

**Key insight:** $L^1$ promotes exact zeros (sparsity) because its unit ball has corners on the axes. $L^2$ shrinks all weights uniformly but never to exactly zero.

## Part 2: Special Matrices (Section 2.6)

| Structure | Defining Property | Why It Matters in ML |
|-----------|-------------------|----------------------|
| Diagonal | $D_{ij}=0$ for $i \ne j$ | $O(n)$ multiply/invert; Adam optimizer uses diagonal preconditioner |
| Symmetric | $A = A^\top$ | Real eigenvalues, orthogonal eigenvectors (spectral theorem); covariance and Hessian are symmetric |
| Orthogonal | $Q^\top Q = I$, $Q^{-1} = Q^\top$ | Preserves norms (no vanishing/exploding gradients); cheap inverse; rotations |
| PD | $\mathbf{x}^\top A \mathbf{x} > 0$ | Hessian PD at critical point = local minimum; unique Cholesky decomposition |
| PSD | $\mathbf{x}^\top A \mathbf{x} \ge 0$ | Covariance matrices are always PSD; valid kernel matrices must be PSD |

**Key insight:** Gram-Schmidt orthogonalization is the constructive proof that orthonormal bases exist. It underpins QR decomposition and shows up whenever we need orthogonal transformations.